# 1. Imports

In [ ]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [ ]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [ ]:
def plot_threat_event(df_threat, show_player_names=True):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal
    - defenders_between_ball_goal
    - total_players_between_ball_goal
    - atk_def_advantage_between_ball_goal

    Espera um DataFrame Spark contendo exatamente um evento.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in attackers] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in defenders] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)
    
    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )
    
    # ============================
    # Contorno do campo
    # ============================

    fig.add_shape(
        type="rect",
        x0=left_x,
        y0=bottom_y,
        x1=right_x,
        y1=top_y,
        line=dict(
            color="black",
            width=2
        ),
        fillcolor="rgba(0,0,0,0)"
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)
    
    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Attacking: {row['eventTeamName']} | "
        f"attackingDirection: {row['attackingDirection']} | "       
        f"Attackers: {row['attackers_between_ball_goal']} | "
        f"Defenders: {row['defenders_between_ball_goal']} | "
        f"Total: {row['total_players_between_ball_goal']} | "
        f"Advantage: {row['atk_def_advantage_between_ball_goal']} | "
        f"Progression: {row['progression_distance']} | "
        f"Threat: {row['threat_score']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
)

    fig.show()

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [ ]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("feature_engineering")
    .getOrCreate()
    )

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [ ]:
# competition_id = 1 (Premier League)
# season = 2022-2023
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [ ]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            #StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        #StructField("visibility", StringType(), True),
        #StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        #StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    #'competitionId',
    #'season',
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    #F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    #F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [ ]:
print('Quantidade de linhas:', df_events.count())

## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [ ]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

#df_games_raw.show()

In [ ]:
df_games_raw.select('venueType').distinct().show()

In [ ]:
df_games_raw.filter(F.col('venueType') == 'NEUTRAL').select('gameId', 'date', 'season', 'venueType', '`team.name`', '`opponentTeam.name`', '`stadium.name`').sort('date').show(truncate=False)

- Apesar de Goodison Park teoricamente ser estádio do Everton e se tratarem de todos os jogos do Everton, o venueType é neutro. Por conta disso, não irei considerar essas partidas.
- Caso necessário usá-las futuramente, podemos ver como ficam os eventos de home e away e se seguem essa estrutura acima mesmo o estádio sendo neutro. Uma sugestão pode ser considerar sempre Everton como casa, mas teria que ver se os eventos de posse ficam de acordo.

In [ ]:
df_games_raw = df_games_raw.filter(F.col('venueType').isin(['TEAM_HOME', 'OPPONENT_HOME']))

In [ ]:
# nesse jogo o liverpool começa do lado direito atacando para a esquerda (https://www.youtube.com/watch?v=Mh82yD4YT6A)
df_games_raw.filter(F.col('gameId') == 4541).show(5)

# nesse jogo o manchester city começa no lado esquerdo atacando para a direita (https://www.youtube.com/watch?v=G1JQc5F-w_g)
df_games_raw.filter(F.col('gameId') == 4452).show(5)

- A variável teamStartSide se refere sempre ao team.name. A questão é saber se esse lado se refere ao lado que o time começa a partida ou se é o sentido do ataque.

- No primeiro jogo (ID 4541), o Liverpool está jogando em casa e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.
- No segundo jogo (ID 4452), o AFC Bournemouth está jogando na casa do adversário Manchester City e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.

- Logo, teamStartSide se refere ao lado no campo que o time começa e o sentido do ataque é na direção oposta (teamStartSide = 'Right', então AttackDirection = 'Left')

In [ ]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw
    .withColumns({
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.id`")).otherwise(F.col("`opponentTeam.id`")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.name`")).otherwise(F.col("`opponentTeam.name`")),
        "homeTeamStartSide": F.when((F.col("venueType") == "TEAM_HOME") & (F.col("teamStartSide") == "Right"), F.col("teamStartSide")).otherwise(F.lit('Left')),
        
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.id`")).otherwise(F.col("`team.id`")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.name`")).otherwise(F.col("`team.name`")),
        "opponentTeamStartSide": F.when((F.col("venueType") == "OPPONENT_HOME") & (F.col("teamStartSide") == "Right"), F.col("teamStartSide")).otherwise(F.lit('Left')),
    })
    .select(
        'gameId',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'), 
        'date',
        'season',
        'venueType',
        #'homeTeamId',
        'homeTeamName',
        #'opponentTeamId',
        'opponentTeamName',
        'homeTeamStartSide',
        'opponentTeamStartSide',
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

#df_games.show(5)

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [ ]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events
    .join(
        df_games, 
        on = "gameId", 
        how='left'
    )
    # filtro para remover os jogos que tinha mandante neutro (venueType == NEUTRAL)
    .filter(~F.col('date').isNull())
)

df_games_events = (
    df_games_events
    # considerar a reversão de lado conforme mudança do primero para o segundo tempo
    # se for primeiro tempo, mantém a variável de StartSide, se não é o contrário
    .withColumns({
        'homeTeamStartSide': F.when(F.col('period') == 1, F.col('homeTeamStartSide')).otherwise(F.col('opponentTeamStartSide')),
        'opponentTeamStartSide': F.when(F.col('period') == 1, F.col('opponentTeamStartSide')).otherwise(F.col('homeTeamStartSide'))
    }) 

    # Sentido do ataque do time é sempre o lado que o outro time começou o período
    .withColumns({
        'homeTeamAttackDirection': F.col('opponentTeamStartSide'),
        'awayTeamAttackDirection': F.col('homeTeamStartSide')
    })
    #.drop('homeTeamStartSide')
)

#df_games_events.show(5)

In [ ]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4452)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4452)).select(filter_cols).show(1)

In [ ]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4541)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4541)).select(filter_cols).show(1)

- Os times mandante e adversário, seus lados na partida e seus sentidos de ataque parecem ter sido definidos corretamente.

In [ ]:
df_games_events.groupby('period').count().show()

In [ ]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

In [ ]:
# window function pra criação do Id de posse
w_pos = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId"
    )
    .orderBy("startGameClock")
)

df_games_events = (
    df_games_events
    .filter(
        # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
        (F.col('period').isin([1,2])) &
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse
    .withColumn(
        "possession_id",
        F.sum(
            F.when(
                F.col("homeTeam") != F.lag("homeTeam").over(w_pos), 1
            ).otherwise(0)
        ).over(
            w_pos.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

## Normalização do ataque sempre pra direita

In [ ]:
# time com a posse está atacando e time sem está defendendo
df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

#df_games_events_tracking.show()

In [ ]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            #p["visibility"].alias("visibility"),
            #p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            #b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        #'attackingDirection',
        #'need_side_revert'
        )
)

#df_games_events_tracking_norm.show(5)

In [ ]:
def euclidean_dist(x1, y1, x2, y2):
    return F.sqrt(
        F.pow(x1 - x2, 2) +
        F.pow(y1 - y2, 2)
    )

In [ ]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

# Menor distância euclidiana entre os escanteios do lado esquerdo até a bola
progression_distance = F.round(
    F.least(
        euclidean_dist(ball_x, ball_y, left_x, top_y),
        euclidean_dist(ball_x, ball_y, left_x, bottom_y)
    ), 2)

## Criação das componentes de ameaça

- Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios do time com a posse até a bola) 
    - Hipótese: Quanto mais o jogador com posse percorrer o campo com a bola na direção do gol, maior a ameaça de gol por estar mais próximo dele.
    - Relação: Diretamente proporcional
- Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
    - Hipótese: Quanto MAIS jogadores dos dois times entre a bola e gol, MENOR a ameaça de gol por haver maior possibilidade de alguma ação defensiva e também por haver chances de um possível chute ser bloqueado.
    - Relação: Inversamente proporcional
- Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)
    - Hipótese: Quanto MAIOR a vantagem numérica do ataque em relação à defesa, MAIOR a ameaça de gol por ter maiores chance de ações ofensivas e menores chances de ações defensivas
    - Relação: Diretamente proporcional

In [ ]:
df_games_events_players_ball_goal = (
    df_games_events_tracking_norm
     .withColumns({
        # Quantidade de jogadores do time mandante entre o gol esquerdo e a bola
        'attackers_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('attackingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        ),
        'defenders_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('defendingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        )
    })
)

df_games_events_players_ball_goal = (
    df_games_events_players_ball_goal.withColumns({
        # Progressão em campo em direção ao gol defendido
        'progression_distance': progression_distance,
        
        # Quantidade absoluta de jogadores entre a bola e o gol
        'total_players_between_ball_goal': F.col('attackers_between_ball_goal') + F.col('defenders_between_ball_goal'),
        
        # Vantagem numérica do ataque em relação à defesa
        'atk_def_advantage_between_ball_goal': F.col('attackers_between_ball_goal') - F.col('defenders_between_ball_goal')
    })
)

# df_games_events_players_ball_goal.show()

### Criação da Ameaça pela Média das 3 componentes normalizadas com Min-Max

In [ ]:
# Obtém mínimos e máximos das componentes
stats = (
    df_games_events_players_ball_goal
    .agg(
        F.min("progression_distance").alias("min_pd"),
        F.max("progression_distance").alias("max_pd"),

        F.min("total_players_between_ball_goal").alias("min_tp"),
        F.max("total_players_between_ball_goal").alias("max_tp"),

        F.min("atk_def_advantage_between_ball_goal").alias("min_adv"),
        F.max("atk_def_advantage_between_ball_goal").alias("max_adv")
    )
    .first()
)

df_threat_final = (
    df_games_events_players_ball_goal
    # Componentes normalizadas [0,1]
    .withColumns({
        # Progressão em campo em direção ao gol defendido
        "progression_distance_norm":
        F.round((F.col("progression_distance") - F.lit(stats["min_pd"])) / F.lit(stats["max_pd"] - stats["min_pd"]), 3),

        # Quantidade absoluta de jogadores entre a bola e o gol (1 - minmax por ser inversamente proporcional)
        "total_players_between_ball_goal_norm": F.round(1 - 
        (F.col("total_players_between_ball_goal") - F.lit(stats["min_tp"])) / F.lit(stats["max_tp"] - stats["min_tp"]), 3), 
        
        # Vantagem numérica do ataque em relação à defesa
        "atk_def_advantage_between_ball_goal_norm":
        F.round((F.col("atk_def_advantage_between_ball_goal") - F.lit(stats["min_adv"])) / F.lit(stats["max_adv"] - stats["min_adv"]), 3),

        # Threat score = média das 3 componentes
        "threat_score": F.round((
            F.col("progression_distance_norm") + F.col("total_players_between_ball_goal_norm") + F.col("atk_def_advantage_between_ball_goal_norm")
        ) / F.lit(3.0), 3)

    })
)

#df_threat_final.show()

In [ ]:
df_threat_final = df_threat_final.localCheckpoint()

In [ ]:
output_path = str(Path().resolve().parent.parent / "data" / "threat_dataset")
df_threat_final.write.mode("overwrite").parquet(output_path)

In [ ]:
# chute do time da casa
event_id = "3a725c404b084914d6f1fef150fb77f9"

df_threat_final.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat_final.filter(F.col("eventId") == event_id))

# attacking players e defending correto mas está atacando pra esquerda

In [ ]:
# clearence do time adversário
event_id = 'c9cf2aa379ba29c9b804597e70e7a202'

df_threat_final.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat_final.filter(F.col("eventId") == event_id))

# attacking e defending team estão corretos mas está atacando pra esquerda

In [ ]:
# cross na posse do time adversário
event_id = '05295389fbfeb8cb3226f4610b8ca26a'

df_threat_final.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat_final.filter(F.col("eventId") == event_id))